In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from chembl_structure_pipeline import standardizer
from rdkit import Chem
from rdkit.Chem import inchi as rd_inchi
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.SaltRemover import SaltRemover

# Constants
VARIANCE_THRESHOLD = 0.2  # Log variation threshold for regression tasks
MAX_MOLECULAR_MASS = 1000  # Maximum molecular mass allowed before duplicate removal

# Instantiate reusable objects once at module level
remover = SaltRemover()
largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
CARBON_PATTERN = Chem.MolFromSmarts('[#6]')  # Used to reject inorganic/loose-salt fragments


def _save_df_if_nonempty(df: pd.DataFrame, path: Path, label: str) -> None:
    """Save a DataFrame to CSV only when it contains rows; print outcome either way."""
    if not df.empty:
        print(f"Saving {label} to '{path}'...")
        df.to_csv(path, index=False)
        print(f"{label.capitalize()} saved successfully.")
    else:
        print(f"No {label} to save.")


def _save_duplicates(savepath: Path, task_type: str,
                     removed_concordant_dup: pd.DataFrame,
                     removed_discordant_dup: pd.DataFrame) -> None:
    """Persist concordant and discordant duplicate records regardless of task type."""
    _save_df_if_nonempty(
        removed_concordant_dup,
        savepath / 'removed_concordant_duplicates.csv',
        'removed concordant duplicates',
    )
    _save_df_if_nonempty(
        removed_discordant_dup,
        savepath / 'removed_discordant_duplicates.csv',
        'removed discordant duplicates',
    )


def prepare_smiles(smile: str) -> str | None:
    """Strip stereochemistry markers and return a canonical SMILES, or None on failure."""
    try:
        cleaned = smile.replace('@', '').replace('/', '').replace('\\', '')
        mol = Chem.MolFromSmiles(cleaned)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, kekuleSmiles=True)
    except Exception as e:
        print(f"Error preparing SMILES '{smile}': {e}")
        return None


def remove_salts(smile: str) -> str | None:
    """Strip salts and reject inorganic/trivially small fragments; return Kekulé SMILES or None."""
    try:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return None
        cleaned_mol = remover.StripMol(mol, dontRemoveEverything=True)
        if cleaned_mol.GetNumAtoms() > 2 and cleaned_mol.HasSubstructMatch(CARBON_PATTERN):
            return Chem.MolToSmiles(cleaned_mol, kekuleSmiles=True)
    except Exception as e:
        print(f"Error processing SMILES '{smile}': {e}")
    return None


def remove_salts_stage(
    df: pd.DataFrame, smiles_col: str
) -> tuple[pd.DataFrame, int, pd.DataFrame]:
    """Remove salts and invalid molecules; return cleaned DataFrame, count removed, and removed rows."""
    print("Removing salts from molecules...")
    df['final_smiles'] = df[smiles_col].apply(remove_salts)
    removed_df = df[df['final_smiles'].isna()].copy()
    df = df.dropna(subset=['final_smiles']).reset_index(drop=True)
    print(f"Removed {len(removed_df)} compounds during salt removal.")
    return df, len(removed_df), removed_df


def _keep_largest_fragment(smile: str) -> str | None:
    """Return the SMILES of the largest fragment, or the unchanged SMILES for single-fragment molecules."""
    try:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return None
        if len(Chem.GetMolFrags(mol)) <= 1:
            return Chem.MolToSmiles(mol, kekuleSmiles=True)
        cleaned_mol = largest_fragment_chooser.choose(mol)
        return Chem.MolToSmiles(cleaned_mol, kekuleSmiles=True) if cleaned_mol is not None else None
    except Exception as e:
        print(f"Error processing mixture '{smile}': {e}")
        return None


def _is_mixture(smile: str) -> bool:
    """Return True when a SMILES string encodes more than one disconnected fragment."""
    mol = Chem.MolFromSmiles(smile)
    return mol is not None and len(Chem.GetMolFrags(mol)) > 1


def remove_mixtures_stage(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, int, pd.DataFrame]:
    """Keep only the largest fragment for multi-component SMILES; return cleaned DataFrame, count, and removed rows."""
    print("Removing molecular mixtures...")
    removed_df = df[df['final_smiles'].apply(_is_mixture)].copy()
    df['final_smiles'] = df['final_smiles'].apply(_keep_largest_fragment)
    df = df.dropna(subset=['final_smiles']).reset_index(drop=True)
    print(f"Removed {len(removed_df)} mixture compounds.")
    return df, len(removed_df), removed_df


def standardize_smiles_stage(
    df: pd.DataFrame, smiles_col: str
) -> tuple[pd.DataFrame, int]:
    """Standardize SMILES via ChEMBL structure pipeline; drop failures."""
    print("Standardizing SMILES...")

    def _standardize(s: str) -> str | None:
        try:
            mol = Chem.MolFromSmiles(s, sanitize=True)
            if mol is None:
                return None
            std_mol = Chem.MolFromMolBlock(standardizer.standardize_molblock(Chem.MolToMolBlock(mol)))
            return Chem.MolToSmiles(std_mol, kekuleSmiles=True)
        except Exception as e:
            print(f"Error standardizing SMILES '{s}': {e}")
            return None

    df['final_smiles'] = df[smiles_col].apply(_standardize)
    removed = int(df['final_smiles'].isna().sum())
    df = df.dropna(subset=['final_smiles']).reset_index(drop=True)
    print(f"Removed {removed} compounds during SMILES standardization.")
    return df, removed


def canonicalize_tautomer_stage(
    df: pd.DataFrame, smiles_col: str = 'final_smiles'
) -> tuple[pd.DataFrame, int]:
    """Collapse tautomeric forms into a canonical SMILES representation."""
    print("Canonicalizing tautomers...")

    def _canonicalize(s: str) -> str | None:
        try:
            mol = Chem.MolFromSmiles(s)
            if mol is None:
                return None
            mol = tautomer_enumerator.Canonicalize(mol)
            Chem.Kekulize(mol, clearAromaticFlags=True)
            return Chem.MolToSmiles(mol, kekuleSmiles=True)
        except Exception as e:
            print(f"Error canonicalizing tautomer of '{s}': {e}")
            return None

    df['final_smiles'] = df[smiles_col].apply(_canonicalize)
    removed = int(df['final_smiles'].isna().sum())
    df = df.dropna(subset=['final_smiles']).reset_index(drop=True)
    print(f"Removed {removed} compounds during tautomer canonicalization.")
    return df, removed


def calculate_inchi_stage(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, int]:
    """Compute InChI strings for duplicate identification; drop failures."""
    print("Calculating InChI for molecules...")

    def _mol_to_inchi(s: str) -> str | None:
        try:
            mol = Chem.MolFromSmiles(s)
            return rd_inchi.MolToInchi(mol) if mol is not None else None
        except Exception as e:
            print(f"Error computing InChI for SMILES '{s}': {e}")
            return None

    df['InChI'] = df['final_smiles'].apply(_mol_to_inchi)
    removed = int(df['InChI'].isna().sum())
    df = df.dropna(subset=['InChI']).reset_index(drop=True)
    print(f"Removed {removed} compounds due to InChI calculation failures.")
    return df, removed


def remove_high_molecular_mass_stage(
    df: pd.DataFrame,
    smiles_col: str = 'final_smiles',
    max_mass: float = MAX_MOLECULAR_MASS,
) -> tuple[pd.DataFrame, int, pd.DataFrame]:
    """Remove molecules whose exact molecular weight exceeds *max_mass*."""
    print(f"Removing molecules with molecular mass above {max_mass}...")

    def _calc_mass(smile: str) -> float | None:
        try:
            mol = Chem.MolFromSmiles(smile)
            return rdMolDescriptors.CalcExactMolWt(mol) if mol is not None else None
        except Exception as e:
            print(f"Error calculating molecular mass of '{smile}': {e}")
            return None

    df['molecular_mass'] = df[smiles_col].apply(_calc_mass)
    removed_df = df[df['molecular_mass'].isna() | (df['molecular_mass'] > max_mass)].copy()
    df = df[df['molecular_mass'].notna() & (df['molecular_mass'] <= max_mass)].reset_index(drop=True)
    print(f"Removed {len(removed_df)} compounds with molecular mass above {max_mass}.")
    return df, len(removed_df), removed_df


def remove_duplicates_regression(
    df: pd.DataFrame,
    target_col: str,
    threshold: float = VARIANCE_THRESHOLD,
    save_log_target: bool = True,
) -> tuple[pd.DataFrame, int, int, pd.DataFrame, pd.DataFrame]:
    """
    Remove duplicates in regression datasets based on log-scale variance.

    - Concordant duplicates (std <= threshold): keep one row with the mean outcome.
    - Discordant duplicates (std > threshold): remove all rows.

    Parameters
    ----------
    df : pd.DataFrame
    target_col : str
        Column with the numeric regression target.
    threshold : float
        Maximum allowed log-scale standard deviation for concordance.
    save_log_target : bool
        When True, retain the 'log_target' column in the returned DataFrame.

    Returns
    -------
    Filtered DataFrame, concordant count, discordant count,
    concordant-removed DataFrame, discordant-removed DataFrame.
    """
    print("Removing duplicates for regression tasks...")

    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

    invalid = int(df[target_col].isna().sum())
    if invalid:
        print(f"Warning: {invalid} non-numeric values found in '{target_col}' — dropping them.")
        df = df.dropna(subset=[target_col]).reset_index(drop=True)

    non_positive = int((df[target_col] <= 0).sum())
    if non_positive:
        print(f"Warning: {non_positive} non-positive values found in '{target_col}' — dropping them.")
        df = df[df[target_col] > 0].reset_index(drop=True)

    df['log_target'] = np.log(df[target_col])

    concordant_removed = 0
    discordant_removed = 0
    keep_rows: list[pd.Series] = []
    removed_concordant_rows: list[pd.DataFrame] = []
    removed_discordant_rows: list[pd.DataFrame] = []

    for _, group in df.groupby('InChI'):
        if len(group) == 1:
            keep_rows.append(group.iloc[0])
            continue
        if group['log_target'].std() <= threshold:
            concordant_removed += len(group) - 1
            representative = group.iloc[0].copy()
            representative[target_col] = group[target_col].mean()
            keep_rows.append(representative)
            removed_concordant_rows.append(group.iloc[1:])
        else:
            discordant_removed += len(group)
            removed_discordant_rows.append(group)

    filtered_df = (
        pd.DataFrame(keep_rows).reset_index(drop=True) if keep_rows
        else pd.DataFrame(columns=df.columns)
    )
    if not save_log_target:
        filtered_df = filtered_df.drop(columns=['log_target'])

    removed_concordant_df = (
        pd.concat(removed_concordant_rows, ignore_index=True) if removed_concordant_rows
        else pd.DataFrame(columns=df.columns)
    )
    removed_discordant_df = (
        pd.concat(removed_discordant_rows, ignore_index=True) if removed_discordant_rows
        else pd.DataFrame(columns=df.columns)
    )

    print(f"Concordant duplicates removed: {concordant_removed}")
    print(f"Discordant duplicates removed: {discordant_removed}")
    return filtered_df, concordant_removed, discordant_removed, removed_concordant_df, removed_discordant_df


def remove_duplicates_classification(
    df: pd.DataFrame, outcome_col: str
) -> tuple[pd.DataFrame, int, int, pd.DataFrame, pd.DataFrame]:
    """
    Remove duplicates in classification datasets.

    - Concordant duplicates (same label): keep one row.
    - Discordant duplicates (conflicting labels): remove all rows.

    Returns
    -------
    Filtered DataFrame, concordant count, discordant count,
    concordant-removed DataFrame, discordant-removed DataFrame.
    """
    print("Removing duplicates for classification tasks...")

    concordant_removed = 0
    discordant_removed = 0
    keep_indices: list[int] = []
    removed_concordant_rows: list[pd.DataFrame] = []
    removed_discordant_rows: list[pd.DataFrame] = []

    for _, group in df.groupby('InChI'):
        if len(group) == 1:
            keep_indices.append(group.index[0])
            continue
        if group[outcome_col].nunique() == 1:
            concordant_removed += len(group) - 1
            keep_indices.append(group.index[0])
            removed_concordant_rows.append(group.iloc[1:])
        else:
            discordant_removed += len(group)
            removed_discordant_rows.append(group)

    final_df = df.loc[keep_indices].reset_index(drop=True)

    removed_concordant_df = (
        pd.concat(removed_concordant_rows, ignore_index=True) if removed_concordant_rows
        else pd.DataFrame(columns=df.columns)
    )
    removed_discordant_df = (
        pd.concat(removed_discordant_rows, ignore_index=True) if removed_discordant_rows
        else pd.DataFrame(columns=df.columns)
    )

    print(f"Concordant duplicates removed: {concordant_removed}")
    print(f"Discordant duplicates removed: {discordant_removed}")
    return final_df, concordant_removed, discordant_removed, removed_concordant_df, removed_discordant_df


def write_log(savepath: Path, log_data: dict) -> None:
    """Persist the curation process log as a plain-text file."""
    log_path = savepath / 'curation_log.txt'
    try:
        print(f"Saving log to '{log_path}'...")
        with log_path.open('w') as f:
            f.write('Curation process log:\n')
            for key, value in log_data.items():
                f.write(f'{key}: {value}\n')
        print("Log saved successfully.")
    except Exception as e:
        print(f"Error saving log: {e}")


def curate_dataset(
    df: pd.DataFrame,
    smiles_col: str,
    outcome_col: str,
    task_type: str = 'classification',
    savepath: str = 'curated_data',
) -> tuple[pd.DataFrame | None, dict | None]:
    """
    Run the full curation pipeline on *df*.

    Steps
    -----
    1. Drop NaN in SMILES / outcome columns.
    2. Remove stereochemistry and validate SMILES.
    3. Standardize SMILES (ChEMBL pipeline).
    4. Remove salts.
    5. Remove mixtures.
    6. Canonicalize tautomers.
    7. Compute InChI.
    8. Remove molecules above MAX_MOLECULAR_MASS.
    9. Remove duplicates (classification or regression).

    Parameters
    ----------
    df : pd.DataFrame
    smiles_col : str
    outcome_col : str
    task_type : {'classification', 'regression'}
    savepath : str
        Directory where curated files and logs will be written.

    Returns
    -------
    Curated DataFrame and log dictionary, or (None, None) on error.
    """
    try:
        log_data: dict = {}
        out_dir = Path(savepath)
        out_dir.mkdir(parents=True, exist_ok=True)

        initial_count = len(df)
        log_data['Initial Compounds'] = initial_count
        print(f"Initial compound count: {initial_count}")

        # Drop rows with missing SMILES or outcome
        print("Removing rows with null SMILES or outcome...")
        df = df.dropna(subset=[smiles_col, outcome_col]).reset_index(drop=True)
        removed_nan = initial_count - len(df)
        log_data['Removed After Drop NaN Molecule/Outcome'] = removed_nan
        print(f"Removed {removed_nan} compounds due to NaN values.")

        # Strip stereochemistry and validate SMILES
        print("Preparing SMILES (removing stereochemistry and validating)...")
        df['final_smiles'] = df[smiles_col].apply(prepare_smiles)
        removed_invalid = int(df['final_smiles'].isna().sum())
        df = df.dropna(subset=['final_smiles']).reset_index(drop=True)
        log_data['Invalid SMILES Removed During Preparation'] = removed_invalid
        print(f"Removed {removed_invalid} compounds with invalid SMILES.")
        print(f"Compounds after SMILES preparation: {len(df)}")

        # Standardize
        df, removed = standardize_smiles_stage(df, 'final_smiles')
        log_data['SMILES Standardized Removed'] = removed
        print(f"Compounds after SMILES standardization: {len(df)}")

        # Remove salts
        df, removed_salts, removed_salts_df = remove_salts_stage(df, 'final_smiles')
        log_data['Salts Removed'] = removed_salts
        print(f"Compounds after salt removal: {len(df)}")
        _save_df_if_nonempty(removed_salts_df, out_dir / 'removed_salts.csv', 'removed salts')

        # Remove mixtures
        df, removed_mixtures, removed_mixtures_df = remove_mixtures_stage(df)
        log_data['Mixtures Removed'] = removed_mixtures
        print(f"Compounds after mixture removal: {len(df)}")
        _save_df_if_nonempty(removed_mixtures_df, out_dir / 'removed_mixtures.csv', 'removed mixtures')

        # Canonicalize tautomers
        df, removed = canonicalize_tautomer_stage(df, 'final_smiles')
        log_data['Tautomer Canonicalization Removed'] = removed
        print(f"Compounds after tautomer canonicalization: {len(df)}")

        # Compute InChI
        df, removed = calculate_inchi_stage(df)
        log_data['InChI Calculation Removed'] = removed
        print(f"Compounds after InChI calculation: {len(df)}")

        # Filter by molecular mass
        df, removed_mass, removed_mass_df = remove_high_molecular_mass_stage(
            df, 'final_smiles', max_mass=MAX_MOLECULAR_MASS
        )
        log_data['High Molecular Mass Removed'] = removed_mass
        print(f"Compounds after molecular mass filter: {len(df)}")
        _save_df_if_nonempty(removed_mass_df, out_dir / 'removed_high_molecular_mass.csv', 'removed high-mass molecules')

        # Remove duplicates
        if task_type == 'classification':
            df, concordant, discordant, removed_conc, removed_disc = remove_duplicates_classification(df, outcome_col)
        elif task_type == 'regression':
            df, concordant, discordant, removed_conc, removed_disc = remove_duplicates_regression(df, outcome_col)
        else:
            raise ValueError(f"task_type must be 'classification' or 'regression', got '{task_type}'.")

        log_data['Concordant Duplicates Removed'] = concordant
        log_data['Discordant Duplicates Removed'] = discordant
        print(f"Compounds after duplicate removal ({task_type}): {len(df)}")
        _save_duplicates(out_dir, task_type, removed_conc, removed_disc)

        final_count = len(df)
        log_data['Final Compounds'] = final_count
        print(f"Final compound count: {final_count}")

        # Drop the original SMILES column, keep only final_smiles
        df = df.drop(columns=[smiles_col], errors='ignore')

        curated_csv = out_dir / 'curated_dataset.csv'
        print(f"Saving curated dataset to '{curated_csv}'...")
        df.to_csv(curated_csv, index=False)
        print("Curated dataset saved successfully.")

        write_log(out_dir, log_data)
        return df, log_data

    except Exception as e:
        print(f'Error during dataset curation: {e}')
        return None, None


[11:30:07] Initializing Normalizer


In [2]:
# Load the dataset
df = pd.read_csv('odorant_odorless.csv')
df

,SMILES,Label
0,C(C(C1C(=C(C(=O)O1)O)O)O)O,1
1,C(C(C(=O)O)N)S,1
2,CC1=C(SC=[N+]1CC2=CN=C(N=C2N)C)CCO.Cl.[Cl-],1
3,CC(C)(C)O,1
4,CCOCC,1
...,...,...
5088,[Al],0
5089,[Tl],0
5090,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,0
5091,[As],0


In [3]:
curated_df, log = curate_dataset(
    df,
    smiles_col='SMILES',
    outcome_col='Label',
    task_type='classification',
    savepath='./',
)

[11:30:07] SMILES Parse Error: syntax error while parsing: vCC(C)C(=O)C(=O)[O-].[Na+]
[11:30:07] SMILES Parse Error: check for mistakes around position 1:
[11:30:07] vCC(C)C(=O)C(=O)[O-].[Na+]
[11:30:07] ^
[11:30:07] SMILES Parse Error: Failed parsing SMILES 'vCC(C)C(=O)C(=O)[O-].[Na+]' for input: 'vCC(C)C(=O)C(=O)[O-].[Na+]'
[11:30:07] SMILES Parse Error: Failed parsing SMILES 'OGJATLSJIMPQBD-UHFFFAOYSA-N' for input: 'OGJATLSJIMPQBD-UHFFFAOYSA-N'
[11:30:07] SMILES Parse Error: syntax error while parsing: ZDQWESQEGGJUCH-UHFFFAOYSA-N
[11:30:07] SMILES Parse Error: check for mistakes around position 1:
[11:30:07] ZDQWESQEGGJUCH-UHFFFAOYSA-N
[11:30:07] ^
[11:30:07] SMILES Parse Error: Failed parsing SMILES 'ZDQWESQEGGJUCH-UHFFFAOYSA-N' for input: 'ZDQWESQEGGJUCH-UHFFFAOYSA-N'
[11:30:07] SMILES Parse Error: syntax error while parsing: LTMQZVLXCLQPCT-UHFFFAOYSA-N
[11:30:07] SMILES Parse Error: check for mistakes around position 1:
[11:30:07] LTMQZVLXCLQPCT-UHFFFAOYSA-N
[11:30:07] ^
[11:30:0

Initial compound count: 5093
Removing rows with null SMILES or outcome...
Removed 3 compounds due to NaN values.
Preparing SMILES (removing stereochemistry and validating)...


[11:30:07] SMILES Parse Error: syntax error while parsing: decasodium;3-methoxy-6-[2-(6-methoxy-4,5-disulfonatooxyoxan-3-yl)oxy-5-[5-(5-methoxy-3,4-disulfonatooxyoxan-2-yl)oxy-3,4-disulfonatooxyoxan-2-yl]oxy-4-sulfonatooxyoxan-3-yl]oxy-4,5-disulfonatooxyoxane-2-carboxylate
[11:30:07] SMILES Parse Error: check for mistakes around position 1:
[11:30:07] decasodium;3-methoxy-6-[2-(6-methoxy-4,5-
[11:30:07] ^
[11:30:07] SMILES Parse Error: Failed parsing SMILES 'decasodium;3-methoxy-6-[2-(6-methoxy-4,5-disulfonatooxyoxan-3-yl)oxy-5-[5-(5-methoxy-3,4-disulfonatooxyoxan-2-yl)oxy-3,4-disulfonatooxyoxan-2-yl]oxy-4-sulfonatooxyoxan-3-yl]oxy-4,5-disulfonatooxyoxane-2-carboxylate' for input: 'decasodium;3-methoxy-6-[2-(6-methoxy-4,5-disulfonatooxyoxan-3-yl)oxy-5-[5-(5-methoxy-3,4-disulfonatooxyoxan-2-yl)oxy-3,4-disulfonatooxyoxan-2-yl]oxy-4-sulfonatooxyoxan-3-yl]oxy-4,5-disulfonatooxyoxane-2-carboxylate'
[11:30:07] WARNING: not removing hydrogen atom without neighbors
[11:30:07] WARNING: not remo

Removed 7 compounds with invalid SMILES.
Compounds after SMILES preparation: 5083
Standardizing SMILES...


[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharger
[11:30:07] Running Normalizer
[11:30:07] Running Uncharg

...

Removed 86 mixture compounds.
Compounds after mixture removal: 4853
Saving removed mixtures to 'removed_mixtures.csv'...
Removed mixtures saved successfully.
Canonicalizing tautomers...


[11:30:12] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:30:14] Tautomer enumeration stopped at 741 tautomers: max transforms reached
[11:30:15] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:30:15] Tautomer enumeration stopped at 1000 tautomers: max tautomers reached
[11:30:16] Tautomer enumeration stopped at 716 tautomers: max transforms reached
[11:30:17] Tautomer enumeration stopped at 359 tautomers: max transforms reached
[11:30:17] Tautomer enumeration stopped at 168 tautomers: max transforms reached
[11:30:18] Tautomer enumeration stopped at 168 tautomers: max transforms reached
[11:30:19] Tautomer enumeration stopped at 716 tautomers: max transforms reached
[11:30:20] Tautomer enumeration stopped at 716 tautomers: max transforms reached
[11:30:20] Tautomer enumeration stopped at 168 tautomers: max transforms reached
[11:30:20] Tautomer enumeration stopped at 162 tautomers: max transforms reached
[11:30:21] Can't kekulize mo

Removed 0 compounds during tautomer canonicalization.
Compounds after tautomer canonicalization: 4853
Calculating InChI for molecules...


[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefined stereo

[11:30:24] WARNING: Omitted undefi

Removed 0 compounds due to InChI calculation failures.
Compounds after InChI calculation: 4853
Removing molecules with molecular mass above 1000...
Removed 13 compounds with molecular mass above 1000.
Compounds after molecular mass filter: 4840
Saving removed high-mass molecules to 'removed_high_molecular_mass.csv'...
Removed high-mass molecules saved successfully.
Removing duplicates for classification tasks...
Concordant duplicates removed: 602
Discordant duplicates removed: 37
Compounds after duplicate removal (classification): 4201
Saving removed concordant duplicates to 'removed_concordant_duplicates.csv'...
Removed concordant duplicates saved successfully.
Saving removed discordant duplicates to 'removed_discordant_duplicates.csv'...
Removed discordant duplicates saved successfully.
Final compound count: 4201
Saving curated dataset to 'curated_dataset.csv'...
Curated dataset saved successfully.
Saving log to 'curation_log.txt'...
Log saved successfully.
